# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashfiqmahi/assignment_FLyRank-AI/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 3: Structure Content Archetype Clustering**

Not every page needs the same action strategy. Some pages are high-performing which are worth protecting, some are stale, some get traffic but weak engagement and others barely register in the search result. Rather than using **"refresh score"** finding the performance pattern the page belongs to maybe more fruitful.

I choose this lane beacuse I prefere visualisation of data. I want to discover recurring performance segments in the inventory and map each segment to a clear action: Clustering profiles, charts, and named segments.

This lane uses **structured clustering on metrics and metadata** (impressions, CTR, position, age, engagement, content type). The dataset does not include article text, so this is **not** semantic clustering — and I will not claim that it is.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

Which **content strategy** should apply to each page — or, at portfolio level, which **archetypes deserve attention first**?

**Who acts on the output?**  
A FlyRank content strategist or editor reviewing a client's inventory. Instead of a flat ranked list, they see segment profiles: "these 800 pages look like stale-visible pages; these 200 look like hidden gems."

**What does a wrong call cost?**  
- Labeling a **champion** as "needs rewrite" → wasted editor time and risk of breaking what works  
- Labeling a **stale visible page** as "monitor only" → missed opportunity while impressions still exist  
- Treating **low-volume noise** as a real segment → false confidence and bad portfolio decisions  

**Why does data/ML help?**  
Many signals interact (visibility, position, CTR, age, freshness, engagement, content type). Hand-written rules handle one pattern at a time; clustering finds **multi-signal groups** that are hard to write as if-statements. The output is a **lens for discovery**, not a single score.

**One-paragraph frame:**  
For a content strategist deciding which performance pattern each page belongs to, I will build **cluster profiles and an action map** from the FlyRank starter dataset (and later the warehouse release), grouping pages by observed search and engagement metrics. Success will be measured by **interpretable cluster separation**, **stability across re-runs**, and whether each segment maps to a sensible action. A wrong segment label costs misallocated editor effort. Plain rules are not enough because pages can be high-impression + low-CTR + old + informational all at once. I will claim only **observed, directional, decision-support** results.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*
Three numbers that make Lane 3 worth pursuing:

1. **Scale:** The starter slice has **30,000** pages across **32** clients — enough inventory to find recurring patterns, not just one-off pages.

2. **Visibility:** **100%** of pages have measurable search visibility (impression tier low/moderate/good/excellent). Clustering on invisible pages would mostly find "no data" — so I will filter to visible pages before clustering.

3. **Diversity:** Log-impression spread has CV = **0.43**, meaning pages are **not** all the same shape. That diversity is what archetype discovery is for — if every page looked identical, clustering would be pointless.

**Early segment hint:** The cross-tab of impression tier × trend direction already suggests different "story types" (e.g. good-impression pages skewing down vs up). Clustering should make these multi-signal groups explicit and visual.

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Number 1: inventory scale ---
n_pages = len(df)
n_clients = df["client_id"].nunique()
print(f"1) Inventory scale: {n_pages:,} pages across {n_clients} clients")

# --- Number 2: enough signal to cluster meaningfully? ---
# Pages with real search visibility (impression tier above 'none')
visible = df[df["impression_tier"].isin(["low", "moderate", "good", "excellent"])]
print(f"2) Pages with measurable search visibility: {len(visible):,} ({len(visible)/len(df):.1%})")

# --- Number 3: behavioral spread — do pages actually look different? ---
# Coefficient of variation on log impressions (high = diverse shapes to discover)
log_impr = np.log1p(df["impressions_90d"])
cv_impr = log_impr.std() / log_impr.mean()
print(f"3) Spread in log(impressions): CV = {cv_impr:.2f} (higher = more segment diversity)")


print(df["content_type"].value_counts(normalize=True).round(3))

pos = df[df["position_tier"] != "no_data"]
print(pos["position_tier"].value_counts(normalize=True).round(3))

print("\n impression × trend cross-tab (raw segment hints) ")
ct = pd.crosstab(df["impression_tier"], df["trend_direction"], normalize="index").round(3)
print(ct)

1) Inventory scale: 30,000 pages across 32 clients
2) Pages with measurable search visibility: 30,000 (100.0%)
3) Spread in log(impressions): CV = 0.43 (higher = more segment diversity)
content_type
keyword article       0.907
feedly article        0.070
comparison article    0.023
Name: proportion, dtype: float64
position_tier
page_1      0.394
striking    0.243
page_3_5    0.241
top_3       0.077
deep        0.044
Name: proportion, dtype: float64

 impression × trend cross-tab (raw segment hints) 
trend_direction   down   flat    new  stable     up
impression_tier                                    
excellent        0.462  0.000  0.001   0.404  0.134
good             0.586  0.000  0.004   0.283  0.127
low              0.454  0.102  0.187   0.111  0.146
moderate         0.615  0.001  0.010   0.213  0.161


##4. Careful words: what I can and can't claim

**What this work CAN say (observed / measured / directional / decision-support):**

- *Observed:* "We observed N recurring performance groups in the inventory when clustering on [feature list]."
- *Measured:* "Cluster 2 has median impressions of X and median CTR of Y% (×100 scale), vs Cluster 4 at A and B."
- *Directional:* "Pages in the stale-visible segment *tended to* have high impressions, older age, and longer time since last update."
- *Decision-support:* "Segment profiles suggest a portfolio-level action map: protect champions, prioritize stale-visible pages for review, monitor low-volume pages."

**What this work CANNOT say:**

- **Not semantic clustering** — no article text in the data; clusters are from metrics/metadata only.
- **Not causal** — clusters describe co-occurring patterns, not "this page type causes ranking."
- **Not Google algorithm discovery** — we grouped observable performance, not reverse-engineered ranking factors.
- **Not guaranteed recovery** — "rewrite" is a suggested review action, not proof that editing fixes traffic.
- **Not ground truth labels** — clusters are a lens; two valid cluster solutions may exist (different K, different features).

**How an editor would use this tomorrow:**  
Open the cluster profile chart → identify which segment dominates their client's inventory → apply the action map for that segment (protect / improve / rewrite / monitor / prune) → use reason codes from cluster centroids, not a opaque score.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.